## Regresion Logistica 

Inicialmente probamos con la regresion logistica, un algoritmo diseñado para algorimtos que son linealmentes separables 

In [3]:
from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned()

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import KFold
import numpy as np

regularizacion = list(np.logspace(-4, 4, 9))

rows = []
for c in regularizacion: 
    accs_train, accs_val, f1s, recalls = [],[],[],[]
    
    kf = KFold(n_splits=5)
    for train_index, val_index in kf.split(X_train): 
        xtrain, xval = X_train[train_index], X_train[val_index]
        ytrain, yval = y_train[train_index], y_train[val_index]
        
        scaler = StandardScaler() 
        xtrain, xval = scaler.fit_transform(xtrain), scaler.transform(xval)
        
        model = LogisticRegression(max_iter=1000000, 
                                        tol=1e-6, 
                                        random_state=0,
                                        C=c)

        model.fit(xtrain, ytrain)
        
        predictions_train = model.predict(xtrain)
        predictions_val = model.predict(xval)
        
        accs_train.append(accuracy_score(ytrain, predictions_train))
        accs_val.append(accuracy_score(yval, predictions_val))
        f1s.append(f1_score(yval, predictions_val))
        recalls.append(recall_score(yval, predictions_val))
        

    rows.append({    
        'Regularizacion C': c,
        'media acc train' : np.mean(accs_train),
        'media acc val': np.mean(accs_val), 
        'IC acc val': np.std(accs_val),
        'media f1' : np.mean(f1s),
        'IC f1': np.std(f1s),
        'media recall': np.mean(recalls),
        'IC recall': np.std(recalls)
    }) 




In [12]:
import pandas as pd 

results = pd.DataFrame(rows)
results.sort_values(by='media f1', ascending=False)

,Regularizacion C,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recall
7,1000.0000,0.948031,0.943000,0.004848,0.943448,0.004097,0.949350,0.004736
8,10000.0000,0.948125,0.942875,0.005117,0.943320,0.004363,0.949103,0.004086
4,1.0000,0.947344,0.942750,0.005025,0.943259,0.004380,0.950319,0.002814
5,10.0000,0.947844,0.942750,0.004483,0.943220,0.003726,0.949566,0.003434
6,100.0000,0.947906,0.942625,0.004409,0.943060,0.003672,0.948824,0.003442
3,0.1000,0.944906,0.940625,0.006212,0.941130,0.005419,0.947326,0.002299
2,0.0100,0.936906,0.935250,0.004377,0.935681,0.003699,0.940793,0.003402
1,0.0010,0.925781,0.924125,0.005163,0.924076,0.004099,0.922146,0.006235
0,0.0001,0.900625,0.900250,0.007380,0.897598,0.007195,0.873713,0.011846


Se obtienen resultados considerablemente buenos cuando se itera con parametro de regularización $c=100$ 

Aunque es muy probable que según los resultados obtenidos el problema de naturaleza lineal, probaremos con diferentes enfoques más allá de una simple linea

Si usamos como nuestra métrica principal el `f1_score` vemos que nuestro mejor modelo es el que usa el kernel `rbf` y valor $c=10$